In [ ]:
!pip install catboost
import pandas as pd
import numpy as np
import re
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

Загрузка данных (чтобы воспроизвести код у себя – нужно загрузить train и test в файлы в среде разработки colab)

In [ ]:
train = pd.read_excel('train.xlsx')
test = pd.read_excel('test.xlsx')

Препроцессинг для Catboost. В итоге обработанные столбцы будут иметь названия на английском языке (чтобы в итоге сформировать пул признаков и точно знать что я их обработал)

In [ ]:
# Количество комнат
for df in [train, test]:
    df['rooms_count'] = df['Количество комнат'].astype(str).str.extract(r'(\d+)').astype(float)
    df['is_apartment'] = df['Количество комнат'].astype(str).str.contains('Аппартаменты').astype(int)
    df['is_penthouse'] = df['Количество комнат'].astype(str).str.contains('Пентхаус').astype(int)
    df['is_separate'] = df['Количество комнат'].astype(str).str.contains('Изолированная|оба варианта', case=False, na=False).astype(int)
    df['is_adjacent'] = df['Количество комнат'].astype(str).str.contains('Смежная|оба варианта', case=False, na=False).astype(int)

# Тип продажи просто переименуем, его нет смысла менять, тк категориальная колонка (catboost без проблем её поймет)
for df in [train, test]:
    df['sale_type'] = df['Тип']

# Метро – вытащим станцию, время и пешком/на машине
for df in [train, test]:
    df['metro_station'] = df['Метро'].astype(str).str.extract(r'м\.\s*([^(]+)')[0].str.strip()
    df['metro_minutes'] = df['Метро'].astype(str).str.extract(r'(\d+)\s*мин').astype(float)
    df['metro_transport'] = df['Метро'].astype(str).str.extract(r'мин\s*(пешком|на машине)')[0]

# Площадь
for df in [train, test]:
    area_split = df['Площадь, м2'].astype(str).str.replace(',', '.').str.split('/', expand=True)
    df['total_area'] = pd.to_numeric(area_split[0], errors='coerce')
    df['living_area'] = pd.to_numeric(area_split[1], errors='coerce') if area_split.shape[1] > 1 else np.nan
    df['kitchen_area'] = pd.to_numeric(area_split[2], errors='coerce') if area_split.shape[1] > 2 else np.nan

# Дом, кол-во этажей и тип дома
for df in [train, test]:
    floor_info = df['Дом'].astype(str).str.extract(r'(\d+)/(\d+)')
    df['floor'] = pd.to_numeric(floor_info[0], errors='coerce')
    df['total_floors'] = pd.to_numeric(floor_info[1], errors='coerce')
    df['house_type'] = df['Дом'].astype(str).str.split(',').str[1].str.strip()

# Парковка (оставляем как есть)
for df in [train, test]:
    df['parking'] = df['Парковка'].fillna('отсутствует')

# Ремонт (оставляем как есть)
for df in [train, test]:
    df['renovation'] = df['Ремонт']

# Площадь комнат (сумма)
def sum_rooms_area(x):
    if pd.isna(x):
        return np.nan
    nums = re.findall(r'[\d.]+', str(x).replace(',', '.'))
    return sum(float(n) for n in nums) if nums else np.nan

train['rooms_area_sum'] = train['Площадь комнат, м2'].apply(sum_rooms_area)
test['rooms_area_sum'] = test['Площадь комнат, м2'].apply(sum_rooms_area)

# Балкон считаем количество и делим лоджия/балкон
for df in [train, test]:
    df['balcony_type'] = df['Балкон'].astype(str).str.extract(r'(Лоджия|Балкон)')[0]
    df['balcony_count'] = df['Балкон'].astype(str).str.extract(r'\((\d+)\)').astype(float).fillna(0)

# Окна (не меняем)

for df in [train, test]:
    df['windows'] = df['Окна']

# Санузел (считаем число раздельных и совмещенных)
for df in [train, test]:
    df['toilet_separate'] = df['Санузел'].astype(str).str.extract(r'Раздельный\s*\((\d+)\)').astype(float).fillna(0)
    df['toilet_combined'] = df['Санузел'].astype(str).str.extract(r'Совмещенный\s*\((\d+)\)').astype(float).fillna(0)

# Название ЖК и год постройки
for df in [train, test]:
    df['complex_name'] = df['Название ЖК'].astype(str).str.split(',').str[0].str.strip()
    df['build_year'] = df['Название ЖК'].astype(str).str.extract(r'Год постройки[-\s]*(\d{4})').astype(float)

# Высота потолков
train['height'] = pd.to_numeric(train['Высота потолков, м'], errors='coerce')
test['height'] = pd.to_numeric(test['Высота потолков, м'], errors='coerce')

# Лифт (количество пассажирских и грузовых), важно, что если этажность больше 5, то по закону лифт обязан стоять
for df in [train, test]:
    df['elevator_pass'] = df['Лифт'].astype(str).str.extract(r'Пасс\s*\((\d+)\)').astype(float).fillna(0)
    df['elevator_cargo'] = df['Лифт'].astype(str).str.extract(r'Груз\s*\((\d+)\)').astype(float).fillna(0)

    # Если этажность больше 5, но лифт не указан, добавляем один пассажирский лифт
    mask_no_elevator = (df['elevator_pass'] == 0) & (df['elevator_cargo'] == 0) & (df['total_floors'] > 5)
    df.loc[mask_no_elevator, 'elevator_pass'] = 1

    # Создаём флаг наличия лифта
    df['has_elevator'] = ((df['elevator_pass'] > 0) | (df['elevator_cargo'] > 0)).astype(int)

# Обрабатываем описание
for df in [train, test]:
    # Количество слов в описании
    df['desc_word_count'] = df['Описание'].astype(str).str.split().str.len()
    # Количество цифр в описании
    df['desc_digit_count'] = df['Описание'].astype(str).str.findall(r'\d').str.len()
    # Количество фотографий
    df['photo_count'] = df['Описание'].astype(str).str.extract(r'(\d+)\s*фото').astype(float).fillna(0)
    # Признаки наличия ключевых слов
    text = df['Описание'].astype(str).str.lower()
    df['desc_has_remont'] = text.str.contains('ремонт|евроремонт|дизайнерский').astype(int)
    df['desc_has_furniture'] = text.str.contains('мебель|меблирована|гарнитур').astype(int)
    df['desc_has_tech'] = text.str.contains('техника|холодильник|стиральная|посудомойка|духовка').astype(int)
    df['desc_has_parking'] = text.str.contains('парковка|паркинг|машиноместо').astype(int)
    df['desc_has_security'] = text.str.contains('охрана|видеонаблюдение|консьерж|домофон').astype(int)
    df['desc_has_balcony'] = text.str.contains('балкон|лоджия').astype(int)
    df['desc_has_view'] = text.str.contains('вид|панорамный|окна').astype(int)

# Мусоропровод
train['trash_chute'] = (train['Мусоропровод'] == 'Да').astype(int)
test['trash_chute'] = (test['Мусоропровод'] == 'Да').astype(int)

# Сведения об условиях (оставляем как есть)
for df in [train, test]:
    df['conditions'] = df['Сведения об условиях']

# Адрес (извлекаем город и улицу)
def parse_address(addr):
    if pd.isna(addr):
        return pd.Series(['Unknown', 'Unknown'])
    parts = str(addr).split(',')
    # Город - первый элемент
    city_part = parts[0].strip()
    # Если первая часть содержит "область", то город - вторая часть
    if 'область' in city_part and len(parts) > 1:
        city = parts[1].strip()
        # Улица - третий элемент, если есть
        street = parts[2].strip() if len(parts) > 2 else 'Unknown'
    else:
        city = city_part
        # Улица - второй элемент, если есть
        street = parts[1].strip() if len(parts) > 1 else 'Unknown'

    # Очищаем улицу от номеров домов и лишней информации
    # Оставляем только название улицы (до первого числа или слова "дом"/"д.")
    street_clean = re.sub(r'\s+\d+.*$', '', street)
    street_clean = re.sub(r',.*$', '', street_clean)
    return pd.Series([city, street_clean])

for df in [train, test]:
    df[['city', 'street']] = df['Адрес'].apply(parse_address)

# Валюта (переводим в рубли)

USD_TO_RUB = 76
EUR_TO_RUB = 90

# Пересчитываем стоимость в рубли для train
def convert_to_rub(row):
    if pd.isna(row['Стоимость']):
        return row['Стоимость']

    currency = row['Валюта'] if pd.notna(row['Валюта']) else 'RUB'
    price = row['Стоимость']

    if currency == 'USD':
        return price * USD_TO_RUB
    elif currency == 'EUR':
        return price * EUR_TO_RUB
    else:  # RUB или другое
        return price

# Применяем конвертацию к train
train['Стоимость'] = train.apply(convert_to_rub, axis=1)

Выбор колонок, заполнение пропусков и обучение на лограрифме цены с помощью Catboost

In [ ]:
feature_columns = [
    'rooms_count', 'is_apartment', 'is_penthouse', 'is_separate', 'is_adjacent',
    'sale_type', 'parking', 'renovation',
    'metro_station', 'metro_minutes', 'metro_transport',
    'total_area', 'living_area', 'kitchen_area',
    'floor', 'total_floors', 'house_type',
    'rooms_area_sum',
    'balcony_type', 'balcony_count',
    'windows',
    'toilet_separate', 'toilet_combined',
    'complex_name', 'build_year',
    'height',
    'elevator_pass', 'elevator_cargo', 'has_elevator',
    'trash_chute',
    'conditions',
    'city', 'street',
    'desc_word_count', 'desc_digit_count', 'photo_count',
    'desc_has_remont', 'desc_has_furniture', 'desc_has_tech',
    'desc_has_parking', 'desc_has_security', 'desc_has_balcony', 'desc_has_view'
]



X = train[feature_columns].copy()
y = train['Стоимость'].copy()

X_test = test[feature_columns].copy()

# Числовые признаки
numeric_cols = ['rooms_count', 'metro_minutes', 'total_area', 'living_area', 'kitchen_area',
                'floor', 'total_floors', 'rooms_area_sum', 'balcony_count', 'build_year',
                'height', 'elevator_pass', 'elevator_cargo',
                'toilet_separate', 'toilet_combined',
                'desc_word_count', 'desc_digit_count', 'photo_count']
for col in numeric_cols:
    if col in X.columns:
        median_val = X[col].median()
        X[col] = X[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)

# Категориальные признаки
cat_cols = ['sale_type', 'parking', 'renovation', 'metro_station', 'metro_transport',
            'house_type', 'balcony_type', 'complex_name',
            'conditions', 'city', 'street','windows']

for col in cat_cols:
    if col in X.columns:
        X[col] = X[col].fillna('Unknown')
        X_test[col] = X_test[col].fillna('Unknown')

# Бинарные признаки
bin_cols = ['is_apartment', 'is_penthouse', 'is_separate', 'is_adjacent',
            'trash_chute', 'has_elevator',
            'desc_has_remont', 'desc_has_furniture', 'desc_has_tech',
            'desc_has_parking', 'desc_has_security', 'desc_has_balcony', 'desc_has_view']
for col in bin_cols:
    X[col] = X[col].fillna(0).astype(int)
    X_test[col] = X_test[col].fillna(0).astype(int)


# Логарифмируем Y

y_log = np.log(y)

print("Обучение на логарифмах цен")
print(f"Диапазон исходных цен (train): {y.min()/1e6:.1f} - {y.max()/1e6:.1f} млн ₽")
print(f"Диапазон логарифмов: {y_log.min():.2f} - {y_log.max():.2f}")
# Модель Catboost, настройки были подобраны вручную с помощью проверки на валидационной выборке, после чего перенес их и обучил модель на всём датасете
model_log = CatBoostRegressor(
    iterations=2000, # На валидационной выборке минимальная ошибка достигалась примерно на 2000 итерации, поэтому оставил именно такой параметр на финальную модель
    learning_rate=0.05,
    depth=7,
    loss_function='RMSE',
    eval_metric='MAE',
    random_seed=71,
    cat_features=cat_cols,
    early_stopping_rounds=200,
    verbose=100,
    border_count=256,
    l2_leaf_reg=5,
    bagging_temperature=1,
    grow_policy='SymmetricTree',
    boosting_type='Plain',
    leaf_estimation_method='Newton',
    leaf_estimation_iterations=3
)

print("\nОбучение модели на логарифмах...")
model_log.fit(X, y_log, verbose=100)

# Предсказания в логарифмическом масштабе
y_test_pred_log = model_log.predict(X_test)

# Обратное преобразование (экспонента)
y_test_pred = np.exp(y_test_pred_log)

results = pd.DataFrame({
    'id': range(len(y_test_pred)),
    'Стоимость': y_test_pred.astype(int)
})

results.to_csv('predictions.csv', index=False)
print("\nПредсказания сохранены в predictions.csv")
print(f"Файл содержит {len(results)} записей")

Обучение на логарифмах цен
Диапазон исходных цен (train): 1.1 - 630.0 млн ₽
Диапазон логарифмов: 13.96 - 20.26

Обучение модели на логарифмах...
0:	learn: 0.8819112	total: 26.7ms	remaining: 53.4s
100:	learn: 0.1801023	total: 2.13s	remaining: 40.1s
200:	learn: 0.1478363	total: 4.16s	remaining: 37.2s
300:	learn: 0.1231094	total: 6.32s	remaining: 35.7s
400:	learn: 0.1067439	total: 8.9s	remaining: 35.5s
500:	learn: 0.0936315	total: 12.3s	remaining: 36.7s
600:	learn: 0.0829675	total: 14.7s	remaining: 34.2s
700:	learn: 0.0742821	total: 18.6s	remaining: 34.5s
800:	learn: 0.0668712	total: 20.8s	remaining: 31.1s
900:	learn: 0.0603115	total: 23.1s	remaining: 28.2s
1000:	learn: 0.0551922	total: 26.4s	remaining: 26.3s
1100:	learn: 0.0503799	total: 28.5s	remaining: 23.2s
1200:	learn: 0.0460915	total: 30.6s	remaining: 20.3s
1300:	learn: 0.0425188	total: 32.7s	remaining: 17.6s
1400:	learn: 0.0392042	total: 34.8s	remaining: 14.9s
1500:	learn: 0.0359827	total: 38s	remaining: 12.6s
1600:	learn: 0.033157